In [1]:
from typing import Annotated, Sequence, TypedDict
from dotenv import load_dotenv
from langchain_core.messages import BaseMessage
from langchain_core.messages import ToolMessage
from langchain_core.messages import SystemMessage
from langchain_core.messages import HumanMessage
from langchain_openai import ChatOpenAI
from langchain_core.tools import tool
from langgraph.graph.message import add_messages
from langgraph.graph import StateGraph, END
from langgraph.prebuilt import ToolNode
import requests
import os
import logging

load_dotenv()

# --- Setup logger for Markdown output ---
LOG_FILE = "agent_tools.md"

class MarkdownFileHandler(logging.FileHandler):
    def emit(self, record):
        msg = self.format(record)
        with open(self.baseFilename, "a", encoding="utf-8") as f:
            f.write(msg + "\n\n")  # extra space for readability


LOG = logging.getLogger("AgentToolsLogger")
LOG.setLevel(logging.DEBUG)

# Markdown log format
formatter = logging.Formatter(
    fmt="### %(asctime)s — **%(levelname)s**\n%(message)s",
    datefmt="%Y-%m-%d %H:%M:%S"
)

if not LOG.handlers:
    md_handler = MarkdownFileHandler(LOG_FILE, mode="a", encoding="utf-8")
    md_handler.setFormatter(formatter)
    LOG.addHandler(md_handler)


def log_code_block(title: str, code: str):
    """Helper to log code snippets in markdown fenced blocks."""
    LOG.debug(f"#### {title}\n```python\n{code}\n```")

In [2]:
@tool
def execute_on_server(code: str) -> dict:
    """
    Sends a Python code snippet to the FastAPI server and returns the JSON response.
    :parameters: Code (str): The Python code to execute.
    :returns: dict: The server's JSON response (contains 'output' or 'error').
    """
    server_url = os.getenv("FASTAPI_ENDPOINT")
    payload = {"code": code}
    headers = {"Content-Type": "application/json"}

    log_code_block("execute_on_server called with code", code)

    try:
        response = requests.post(server_url, json=payload, headers=headers)
        response.raise_for_status()
        result = response.json()
        LOG.debug(f"**Response:**\n```json\n{result}\n```")
        return result
    except requests.RequestException as e:
        LOG.error(f"**Error:** {e}")
        return {"error": f"Request failed: {e}"}


@tool
def generate_code(prompt: str) -> str:
    """
    Generates a Python code snippet that if ran on the FastAPI server answers the given question.
    The code to be generated prompt must never ask for the data to be mutated on the server,
    only query and analyze data.
    :parameters: Prompt (str): The question to answer.
    :returns: str: The generated code snippet.
    """
    return """git_project = graph_data["git"]
num_commits = len(git_project.git_commit_registry.all)
project_name = git_project.name

print(f"Project '{project_name}' has {num_commits} commits.")"""
    LOG.debug(f"**Prompt received:** {prompt}")

    code_generation_llm = ChatOpenAI(
        model=os.getenv("OPENAI_MODEL_FOR_CODE"),
        max_retries=2
    )

    guidelines_text = None
    try:
        with open("../code_generation_guidelines.txt", "r", encoding="utf-8") as f:
            guidelines_text = f.read()
            LOG.debug("_guidelines.txt loaded successfully_")
    except FileNotFoundError:
        LOG.warning("_guidelines.txt not found at ../guidelines.txt_")
    except Exception as e:
        LOG.error(f"_Error reading code_generation_guidelines.txt: {e}_")

    messages = []
    if guidelines_text:
        messages.append(("system", guidelines_text))
    messages.append(("user", prompt))

    try:
        response = code_generation_llm.invoke(messages)
        log_code_block("Generated code", response.content)
        return response.content
    except Exception as e:
        LOG.error(f"_LLM invocation failed: {e}_")
        return f"# ERROR: Failed to generate code ({e})"

tools = [execute_on_server, generate_code]

In [3]:
class AgentState(TypedDict):
    messages: Annotated[Sequence[BaseMessage], add_messages]

oracle = ChatOpenAI(model=os.getenv("OPENAI_MODEL_FOR_ORACLE")).bind_tools(tools)

def ask_oracle(state: AgentState) -> AgentState:
    guidelines_text = None
    try:
        with open("../oracle_prompt.txt", "r", encoding="utf-8") as f:
            guidelines_text = f.read()
            LOG.debug("oracle_prompt.txt loaded successfully_")
    except FileNotFoundError:
        LOG.warning("_oracle_prompt.txt not found at ../oracle_prompt.txt_")
    except Exception as e:
        LOG.error(f"_Error reading oracle_prompt.txt: {e}_")

    system_prompt = SystemMessage(content=guidelines_text)

    response = oracle.invoke([system_prompt, *state["messages"]])

    LOG.debug(f"**Oracle messages:** {response}")

    return {"messages" : [response]}

def should_continue(state :AgentState):
    messages = state["messages"]
    last_message = messages[-1]
    if not last_message.tool_calls:
        return "end"
    return "continue"

In [4]:
graph = StateGraph(AgentState)

graph.add_node("Oracle", ask_oracle)

tool_node = ToolNode(tools = tools)
graph.add_node("Tools", tool_node)

graph.set_entry_point("Oracle")
graph.add_conditional_edges(
    "Oracle",
    should_continue,
    {
        "continue": "Tools",
        "end": END
    }
)
graph.add_edge("Tools", "Oracle")

app = graph.compile()

In [5]:
inputs = {"messages" :[HumanMessage("Care este numarul de commits din evdenta proiectului evaluat acum?")]}
result = app.invoke(inputs)

In [6]:
for message in result["messages"]:
    message.pretty_print()

================================ Human Message =================================

What si the number of GitCommit s from the repo we are analysing now
================================== Ai Message ==================================

tool_decision: generate_code+execute
Reason: You asked for a concrete statistic from the project graph (number of GitCommit entities), which requires querying the server’s data. I will generate read-only code to count commits for the current repository context, and if that context is unavailable, I’ll provide per-repo counts and a total.

Proceeding to generate code that:
- Counts GitCommit entities for the active/current repository context if available.
- Falls back to counts per repository and a grand total if no active context is set.
- Returns a small JSON with active_repo, commit_count, per_repo, and total_commits.
Tool Calls:
  generate_code (call_pihWEYrIBeLE9DUGPmxUryin)
 Call ID: call_pihWEYrIBeLE9DUGPmxUryin
  Args:
    prompt: Count the total num